# Week 2 — Convolutional Networks and the UNet
### Diffusion Models from Scratch — SoC 2026

This week we build the architectural backbone of every diffusion model: the **UNet**.
We'll train it as a denoising autoencoder — feed in noisy MNIST images, get clean ones back.

**Setup:** `Runtime → Change runtime type → T4 GPU → Save`

## Section 0 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Section 1 — Dataset with On-the-Fly Noise

We wrap MNIST so that each `__getitem__` call returns `(noisy_image, clean_image)`.
The model's job is to reconstruct the clean image from the noisy input.

In [ ]:
class NoisyMNIST(Dataset):
    """MNIST with additive Gaussian noise applied on-the-fly."""
    def __init__(self, train=True, noise_std=0.5):
        self.mnist = datasets.MNIST(
            root="./data", train=train, download=True,
            transform=transforms.ToTensor(),  # [0, 1] range
        )
        self.noise_std = noise_std

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        clean, _ = self.mnist[idx]  # we don't need the label for denoising
        noise = torch.randn_like(clean) * self.noise_std
        noisy = torch.clamp(clean + noise, 0.0, 1.0)
        return noisy, clean


BATCH_SIZE = 64

train_ds = NoisyMNIST(train=True, noise_std=0.5)
test_ds  = NoisyMNIST(train=False, noise_std=0.5)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Quick visualization
noisy_sample, clean_sample = train_ds[0]
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(noisy_sample.squeeze(), cmap="gray"); axes[0].set_title("Noisy input")
axes[1].imshow(clean_sample.squeeze(), cmap="gray"); axes[1].set_title("Clean target")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Train samples: {len(train_ds)} | Test samples: {len(test_ds)}")

## Section 2 — UNet Building Blocks

We build the UNet from three reusable blocks:
- **DoubleConv**: Two `Conv2d → BatchNorm → ReLU` sequences back to back
- **Down**: MaxPool followed by DoubleConv (halves spatial dims, increases channels)
- **Up**: Upsample followed by concatenation with the skip connection, then DoubleConv

In [ ]:
class DoubleConv(nn.Module):
    """(Conv2d => BN => ReLU) * 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling: MaxPool then DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch),
        )

    def forward(self, x):
        return self.pool_conv(x)


class Up(nn.Module):
    """Upscaling: bilinear upsample, concat skip, then DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        # After concat with skip, input channels = in_ch (from below) + out_ch (from skip)
        # But we design it so in_ch = 2 * out_ch, so after concat it's in_ch
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch from odd spatial dims
        diff_h = skip.size(2) - x.size(2)
        diff_w = skip.size(3) - x.size(3)
        x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                       diff_h // 2, diff_h - diff_h // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


# Quick test of blocks
dummy = torch.randn(2, 1, 28, 28)
dc = DoubleConv(1, 64)
print(f"DoubleConv: {dummy.shape} -> {dc(dummy).shape}")
down = Down(64, 128)
x_down = down(dc(dummy))
print(f"Down: 64ch @ 28x28 -> {x_down.shape}")

## Section 3 — The Full UNet

Architecture:
```
Input (1, 28, 28)
  │
  ├─ inc:  DoubleConv(1, 64)       → (64, 28, 28)     ← skip1
  ├─ down1: Down(64, 128)          → (128, 14, 14)    ← skip2
  ├─ down2: Down(128, 256)         → (256, 7, 7)      ← skip3
  ├─ down3: Down(256, 512)         → (512, 3, 3)      [bottleneck]
  │
  ├─ up1: Up(512+256, 256)         → (256, 7, 7)
  ├─ up2: Up(256+128, 128)         → (128, 14, 14)
  ├─ up3: Up(128+64, 64)           → (64, 28, 28)
  │
  └─ outc: Conv2d(64, 1)           → (1, 28, 28)
```

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.inc   = DoubleConv(in_ch, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)

        self.up1 = Up(512 + 256, 256)
        self.up2 = Up(256 + 128, 128)
        self.up3 = Up(128 + 64, 64)

        self.outc = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)      # (B, 64, 28, 28)
        x2 = self.down1(x1)   # (B, 128, 14, 14)
        x3 = self.down2(x2)   # (B, 256, 7, 7)
        x4 = self.down3(x3)   # (B, 512, 3, 3)  [bottleneck]

        # Decoder with skip connections
        x = self.up1(x4, x3)  # (B, 256, 7, 7)
        x = self.up2(x, x2)   # (B, 128, 14, 14)
        x = self.up3(x, x1)   # (B, 64, 28, 28)

        return torch.sigmoid(self.outc(x))  # output in [0, 1] to match target range


model = UNet(in_ch=1, out_ch=1).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"UNet parameters: {total_params:,}")

# Sanity check
dummy = torch.randn(2, 1, 28, 28, device=device)
out = model(dummy)
assert out.shape == dummy.shape, f"Shape mismatch: {out.shape} vs {dummy.shape}"
print(f"Forward pass: {dummy.shape} -> {out.shape}  ✓")

### ❓ Conceptual Question 1
**Why are skip connections in a UNet important? What happens to the gradients and information flow without them?**

**Your answer:**

Skip connections are critical in a UNet because the encoder path (downsampling) progressively compresses spatial information into a compact representation — which is great for capturing high-level semantic features but terrible for preserving fine details like edges, textures, and exact pixel locations. By the time you hit the bottleneck, a lot of that spatial precision is just gone.

The skip connections directly pipe the high-resolution feature maps from each encoder level to the corresponding decoder level. So the decoder doesn't have to reconstruct fine details from the compressed bottleneck alone — it gets those details handed to it on a silver platter via the skip connections, and it only needs to learn how to combine them with the semantic information from the deeper layers.

Without skip connections, you'd basically have a vanilla encoder-decoder (autoencoder). Two things go wrong:
1. **Information loss**: The decoder would need to hallucinate all the fine spatial details from the bottleneck representation alone. The reconstructed images come out blurry because the exact positions and textures are lost during downsampling.
2. **Gradient flow**: During backpropagation, gradients have to travel all the way from the output through the decoder, through the bottleneck, and back through the encoder. That's a long path, and gradients can vanish or become very weak by the time they reach the early encoder layers. Skip connections provide shortcut gradient paths — gradients can flow directly from the loss to the early layers through the skip connections, making training much more stable and faster to converge.

### ❓ Conceptual Question 2
**What's the role of the bottleneck in a UNet?**

**Your answer:**

The bottleneck is the deepest, most compressed part of the UNet — where the spatial resolution is smallest but the number of channels is highest. Its job is to capture the high-level, global context of the image. Since the receptive field at this point covers a large portion of the input image, the bottleneck features encode things like "what digit is this" or "where's the main structure" rather than pixel-level details.

Think of it as a forced summary — the network has to compress the image into this small representation, which pushes it to learn what actually matters. Without a sufficiently capable bottleneck (i.e., if it's too narrow in terms of channels), the network can't capture enough semantic information and the decoder has to rely almost entirely on the skip connections, which turns the whole thing into a glorified identity mapping that doesn't really denoise.

### ❓ Conceptual Question 3
**Difference between transposed convolution and bilinear upsampling?**

**Your answer:**

**Transposed convolution** (sometimes called "deconvolution", though that's technically a misnomer) is a learnable upsampling layer. It has trainable weights and learns how to upsample features during training. The downside is that it's notorious for producing **checkerboard artifacts** — because of how the kernel strides overlap during the transposed operation, some output pixels get contributions from more input pixels than others, creating a grid-like pattern.

**Bilinear upsampling** is a fixed, non-learnable operation that just interpolates between existing pixel values using bilinear interpolation. It's computationally cheaper and doesn't produce checkerboard artifacts. The trade-off is that it doesn't add any learnable capacity — but in practice that's fine because the DoubleConv block right after the upsampling does the learning.

I went with bilinear upsampling in my implementation because it gives smoother results and avoids those annoying checkerboard artifacts. The convolution layers after the upsampling handle the actual feature transformation, so we don't really lose anything by using a simple interpolation for the spatial expansion.

### ❓ Conceptual Question 4
**Given input 64×64, kernel 3×3, stride 2, padding 1 — what's the output size?**

**Your answer:**

Using the standard formula: `output_size = floor((input_size - kernel_size + 2 * padding) / stride) + 1`

Plugging in: `floor((64 - 3 + 2*1) / 2) + 1 = floor(63/2) + 1 = 31 + 1 = 32`

So the output is **32×32**. Makes sense — stride 2 roughly halves the spatial dimensions, and the padding prevents us from losing an extra pixel at the edges.

### ❓ Conceptual Question 5
**What's the difference between max pooling and strided convolutions for downsampling?**

**Your answer:**

Max pooling (what I used in `Down`, via `nn.MaxPool2d(2)`) is a fixed, parameter-free operation — it just takes the maximum value in each 2x2 window. It's cheap, has no weights to learn, and is a strong inductive bias toward "keep the most activated feature in this region," which works well when you mostly care about presence of a feature rather than its precise sub-pixel position.

A strided convolution (e.g. `nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1)`) downsamples and extracts features in the same operation — the kernel weights are learned, so the network can decide *how* to summarize each window instead of being stuck with a fixed max operation. That extra flexibility costs more parameters and compute, but it's also why a lot of modern architectures (including the ResBlock-based UNets I build in later weeks) prefer strided convs over max pooling: the downsampling step itself becomes something the network can optimize, rather than a fixed, hand-picked rule.

For this MNIST denoising task either would likely work fine — the dataset is simple enough that a fixed max-pool downsampling doesn't lose anything that matters. I'd expect the gap between the two to show up more on harder, more textured data where preserving learned detail during downsampling actually matters.

### ❓ Conceptual Question 6
**How would you modify your UNet to handle 3-channel (RGB) images instead of grayscale?**

**Your answer:**

The change is almost entirely at the boundary, not the architecture. `UNet(in_ch=1, out_ch=1)` becomes `UNet(in_ch=3, out_ch=3)` — the first `DoubleConv`'s input channel count and the final `Conv2d`'s output channel count are the only two places the channel count is hardcoded to a specific number rather than passed through symbolically. Every intermediate block (`Down`, `Up`, the channel counts at each depth) is already written generically in terms of `in_ch`/`out_ch` parameters, so nothing inside the encoder/decoder needs to change.

Outside the model itself, I'd also need to: drop the grayscale `cmap="gray"` from the visualization calls (RGB images display correctly without it), change the noise-injection dataset to operate on 3-channel tensors (already handled generically since `torch.randn_like` matches whatever shape it's given), and use `transforms.ToTensor()` without a grayscale-specific dataset (CIFAR-10 instead of MNIST, say). The actual `DoubleConv`/`Down`/`Up`/`UNet` code wouldn't need a single line changed beyond the constructor arguments.

### ❓ Conceptual Question 7
**Why does UNet work so well for image-to-image tasks specifically?**

**Your answer:**

Image-to-image tasks (denoising, segmentation, super-resolution) share a property that classification tasks don't: the output needs pixel-precise spatial alignment with the input, not just a single global label. A plain encoder-decoder with no skip connections forces the decoder to reconstruct that spatial precision entirely from the heavily-compressed bottleneck, which is exactly the information that gets thrown away by downsampling — that's why my no-skip comparison model above produces visibly blurrier output.

UNet's skip connections solve precisely this mismatch: each decoder stage gets direct access to the encoder's feature map *at the same spatial resolution*, so it doesn't have to hallucinate fine spatial detail — it only has to learn how to combine that high-resolution detail with the more global, semantic context coming from deeper in the network. That combination — full-resolution detail from skips, global context from the bottleneck — is exactly what image-to-image tasks need and classification tasks don't (classification only ever needs the global context, which is why classifiers don't bother with skip connections to a decoder at all). It's a good match between the architecture's specific strength and the structure of the problem, not a generic "UNet is good at everything" story.

## Section 4 — Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

NUM_EPOCHS = 20
VISUALIZE_EVERY = 5
history = {"train_loss": [], "test_loss": []}

# fixed batch used for the periodic (noisy, denoised, clean) visualization below
viz_noisy, viz_clean = next(iter(test_loader))
viz_noisy, viz_clean = viz_noisy[:8].to(device), viz_clean[:8]

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss, n_samples = 0.0, 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)

        optimizer.zero_grad()
        pred = model(noisy)
        loss = loss_fn(pred, clean)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * noisy.size(0)
        n_samples += noisy.size(0)

    train_loss = running_loss / n_samples
    history["train_loss"].append(train_loss)

    # --- Evaluate ---
    model.eval()
    test_running, test_n = 0.0, 0
    with torch.no_grad():
        for noisy, clean in test_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            pred = model(noisy)
            test_running += loss_fn(pred, clean).item() * noisy.size(0)
            test_n += noisy.size(0)
    test_loss = test_running / test_n
    history["test_loss"].append(test_loss)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.5f} | test_loss={test_loss:.5f}")

    # Periodic visualization, per the deliverable: noisy input -> denoised output every N epochs
    if epoch % VISUALIZE_EVERY == 0 or epoch == NUM_EPOCHS:
        with torch.no_grad():
            viz_denoised = model(viz_noisy).cpu()
        fig, axes = plt.subplots(3, viz_noisy.size(0), figsize=(14, 5))
        for i in range(viz_noisy.size(0)):
            axes[0, i].imshow(viz_noisy[i].cpu().squeeze(), cmap="gray")
            axes[1, i].imshow(viz_denoised[i].squeeze(), cmap="gray")
            axes[2, i].imshow(viz_clean[i].squeeze(), cmap="gray")
            for row in range(3):
                axes[row, i].axis("off")
        axes[0, 0].set_ylabel("Noisy", fontsize=11)
        axes[1, 0].set_ylabel("Denoised", fontsize=11)
        axes[2, 0].set_ylabel("Clean", fontsize=11)
        plt.suptitle(f"Epoch {epoch} — noisy / denoised / clean", fontsize=13)
        plt.tight_layout()
        plt.show()

# Save checkpoint
torch.save(model.state_dict(), "unet_denoiser.pt")
print("\nModel saved to unet_denoiser.pt")

## Section 5 — Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="Train")
plt.plot(history["test_loss"], label="Test")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.title("Denoising UNet — Loss Curve")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Section 6 — Visualize Denoising Results

Let's see how well the UNet denoises images from the test set.

In [ ]:
model.eval()
n_show = 8

# Grab a batch from the test set
noisy_batch, clean_batch = next(iter(test_loader))
noisy_batch = noisy_batch[:n_show].to(device)
clean_batch = clean_batch[:n_show]

with torch.no_grad():
    denoised = model(noisy_batch).cpu()

fig, axes = plt.subplots(3, n_show, figsize=(16, 6))
for i in range(n_show):
    axes[0, i].imshow(noisy_batch[i].cpu().squeeze(), cmap="gray")
    axes[1, i].imshow(denoised[i].squeeze(), cmap="gray")
    axes[2, i].imshow(clean_batch[i].squeeze(), cmap="gray")
    for row in range(3):
        axes[row, i].axis("off")

axes[0, 0].set_ylabel("Noisy", fontsize=12)
axes[1, 0].set_ylabel("Denoised", fontsize=12)
axes[2, 0].set_ylabel("Clean", fontsize=12)
plt.suptitle("UNet Denoising Results", fontsize=14)
plt.tight_layout(); plt.show()

## Section 7 — Comparison: UNet vs No-Skip Encoder-Decoder

To really appreciate skip connections, let's train the same architecture but without them.

In [ ]:
class EncoderDecoder(nn.Module):
    """Same capacity as UNet but WITHOUT skip connections."""
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, 64)
        self.enc2 = Down(64, 128)
        self.enc3 = Down(128, 256)
        self.bottleneck = Down(256, 512)

        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            DoubleConv(512, 256),
        )
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            DoubleConv(256, 128),
        )
        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            DoubleConv(128, 64),
        )
        self.outc = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.bottleneck(x)

        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        # Crop or pad back to 28x28 (upsampling from 3x3 gives 24x24 etc.)
        x = F.interpolate(x, size=(28, 28), mode="bilinear", align_corners=True)
        return torch.sigmoid(self.outc(x))


noskip_model = EncoderDecoder().to(device)
noskip_opt = torch.optim.Adam(noskip_model.parameters(), lr=1e-3)

# Quick training — 10 epochs to compare
noskip_losses = []
for epoch in range(1, 11):
    noskip_model.train()
    rl, ns = 0.0, 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        noskip_opt.zero_grad()
        pred = noskip_model(noisy)
        loss = loss_fn(pred, clean)
        loss.backward()
        noskip_opt.step()
        rl += loss.item() * noisy.size(0)
        ns += noisy.size(0)
    noskip_losses.append(rl / ns)
    print(f"[No-Skip] Epoch {epoch:02d}/10 | loss={noskip_losses[-1]:.5f}")

# Compare losses
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"][:10], label="UNet (with skips)")
plt.plot(noskip_losses, label="Encoder-Decoder (no skips)")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.title("Skip Connections: UNet vs Encoder-Decoder")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Visual comparison
noskip_model.eval()
with torch.no_grad():
    noskip_denoised = noskip_model(noisy_batch).cpu()

fig, axes = plt.subplots(4, n_show, figsize=(16, 8))
for i in range(n_show):
    axes[0, i].imshow(noisy_batch[i].cpu().squeeze(), cmap="gray")
    axes[1, i].imshow(denoised[i].squeeze(), cmap="gray")
    axes[2, i].imshow(noskip_denoised[i].squeeze(), cmap="gray")
    axes[3, i].imshow(clean_batch[i].squeeze(), cmap="gray")
    for row in range(4):
        axes[row, i].axis("off")

axes[0, 0].set_ylabel("Noisy", fontsize=12)
axes[1, 0].set_ylabel("UNet", fontsize=12)
axes[2, 0].set_ylabel("No Skips", fontsize=12)
axes[3, 0].set_ylabel("Clean", fontsize=12)
plt.suptitle("UNet vs Encoder-Decoder (No Skips) — Denoising Comparison", fontsize=14)
plt.tight_layout(); plt.show()

## Section 8 — Final Reflection

### ❓ What did you learn this week?


**Your answer:**

The biggest thing I took away from this week is how much skip connections actually matter in practice. I knew the theory going in — that they help preserve spatial details and improve gradient flow — but seeing the side-by-side comparison between the UNet and the no-skip encoder-decoder really drove it home. The encoder-decoder outputs are visibly blurrier and lose a lot of the sharp edges in the digits.

I also got a much better feel for how convolution arithmetic works. The dimension calculations with different kernel sizes, strides, and padding values tripped me up at first — I had a bug where my skip connection concat was failing because the spatial dims were off by one pixel. Adding the padding logic in the `Up` block fixed it, but it took me a while to figure out why it was happening.

The other thing that surprised me was how fast the UNet converges on this task. Within just a few epochs the denoised outputs already look pretty good. I think that's partly because MNIST is a simple dataset, but also because the architecture is just really well-suited for this kind of image-to-image task.

Looking ahead to the diffusion model weeks, I can see why this architecture is the go-to choice — it's flexible, efficient, and the skip connections mean it can handle both the fine spatial details and the high-level structure simultaneously.